In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv('Superstore.csv',encoding='latin')

# Basic inspection
print(df.shape)
print(df.head())
print(df.info())
print(df.describe())

(51290, 24)
   ÿRow ID         Order ID  Order Date   Ship Date     Ship Mode Customer ID  \
0    32298   CA-2012-124891  31-07-2012  31-07-2012      Same Day    RH-19495   
1    26341    IN-2013-77878  05-02-2013  07-02-2013  Second Class    JR-16210   
2    25330    IN-2013-71249  17-10-2013  18-10-2013   First Class    CR-12730   
3    13524  ES-2013-1579342  28-01-2013  30-01-2013   First Class    KM-16375   
4    47221     SG-2013-4320  05-11-2013  06-11-2013      Same Day     RH-9495   

      Customer Name      Segment           City            State  ...  \
0       Rick Hansen     Consumer  New York City         New York  ...   
1     Justin Ritter    Corporate     Wollongong  New South Wales  ...   
2      Craig Reiter     Consumer       Brisbane       Queensland  ...   
3  Katherine Murray  Home Office         Berlin           Berlin  ...   
4       Rick Hansen     Consumer          Dakar            Dakar  ...   

         Product ID    Category Sub-Category  \
0   TEC-AC-100

In [7]:
# Check missing values
df.isnull().sum()

# Drop column with no Postal Code (critical field)
df = df.drop(['Postal Code'],axis=1)

In [8]:
df.duplicated().sum()

df = df.drop_duplicates()

In [9]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

C:\Users\nihal\AppData\Local\Temp\ipykernel_13604\312075123.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Order Date'] = pd.to_datetime(df['Order Date'])
C:\Users\nihal\AppData\Local\Temp\ipykernel_13604\312075123.py:2: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Ship Date'] = pd.to_datetime(df['Ship Date'])


In [10]:
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Day'] = df['Order Date'].dt.day
df['Hour'] = df['Order Date'].dt.hour
df['Weekday'] = df['Order Date'].dt.day_name()

In [11]:
reference_date = df['Order Date'].max() + pd.Timedelta(days=1)

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 28 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ÿRow ID         51290 non-null  int64         
 1   Order ID        51290 non-null  object        
 2   Order Date      51290 non-null  datetime64[ns]
 3   Ship Date       51290 non-null  datetime64[ns]
 4   Ship Mode       51290 non-null  object        
 5   Customer ID     51290 non-null  object        
 6   Customer Name   51290 non-null  object        
 7   Segment         51290 non-null  object        
 8   City            51290 non-null  object        
 9   State           51290 non-null  object        
 10  Country         51290 non-null  object        
 11  Market          51290 non-null  object        
 12  Region          51290 non-null  object        
 13  Product ID      51290 non-null  object        
 14  Category        51290 non-null  object        
 15  Su

In [17]:
rfm = df.groupby('Customer ID').agg({
    'Order Date': lambda x: (reference_date - x.max()).days,  # Recency
    'Order ID': 'nunique',                                   # Frequency
    'Sales': 'sum'                                       # Monetary
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']

In [18]:
# Recency (lower is better → reverse labels)
rfm['R_score'] = pd.qcut(
    rfm['Recency'],
    q=4,
    labels=[4,3,2,1],
    duplicates='drop'
)

# Frequency (higher is better)
rfm['F_score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    q=4,
    labels=[1,2,3,4]
)

# Monetary (higher is better)
rfm['M_score'] = pd.qcut(
    rfm['Monetary'].rank(method='first'),
    q=4,
    labels=[1,2,3,4]
)

In [19]:
df_cleaned = df.copy()
df_cleaned.head()

,ÿRow ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,City,State,...,Quantity,Discount,Profit,Shipping Cost,Order Priority,Year,Month,Day,Hour,Weekday
0,32298,CA-2012-124891,2012-07-31,2012-07-31,Same Day,RH-19495,Rick Hansen,Consumer,New York City,New York,...,7,0.0,762.1845,933.57,Critical,2012,7,31,0,Tuesday
1,26341,IN-2013-77878,2013-02-05,2013-02-07,Second Class,JR-16210,Justin Ritter,Corporate,Wollongong,New South Wales,...,9,0.1,-288.7650,923.63,Critical,2013,2,5,0,Tuesday
2,25330,IN-2013-71249,2013-10-17,2013-10-18,First Class,CR-12730,Craig Reiter,Consumer,Brisbane,Queensland,...,9,0.1,919.9710,915.49,Medium,2013,10,17,0,Thursday
3,13524,ES-2013-1579342,2013-01-28,2013-01-30,First Class,KM-16375,Katherine Murray,Home Office,Berlin,Berlin,...,5,0.1,-96.5400,910.16,Medium,2013,1,28,0,Monday
4,47221,SG-2013-4320,2013-11-05,2013-11-06,Same Day,RH-9495,Rick Hansen,Consumer,Dakar,Dakar,...,8,0.0,311.5200,903.04,Critical,2013,11,5,0,Tuesday


In [20]:
df_cleaned.to_csv("cleaned_superstore.csv", index=False)
rfm.to_csv("rfm_customers.csv")